In [1]:
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, Subset, DataLoader
import numpy as np

from dataset import create_dataloaders
from tcn_model import MEGTCN
from gan_model import MEGGAN
from cnn_baseline_1d import CNNBaseline1D
from train import train_one_epoch
from evaluate import evaluate
from grid_search import run_grid_search

In [2]:
DATA_DIR = "preprocessed_data/Intra"
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-3
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
intra_train_loader, intra_test_loader = create_dataloaders(DATA_DIR, BATCH_SIZE, add_person_id=True)

Loading test data...
Loading train data...
Loaded 32 training samples
Loaded 8 test samples
Class distribution in training: [8 8 8 8]
Class distribution in test: [2 2 2 2]
Train batches: 4
Test batches: 1


### Cross Dataset

In [4]:
DATA_DIR = "preprocessed_data/Cross"
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-3
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

cross_train_loader, cross_test_loader = create_dataloaders(DATA_DIR, BATCH_SIZE, add_person_id=True)

Using device: cuda
Loading test1 data...
Loading test2 data...
Loading test3 data...
Loading train data...
Loaded 64 training samples
Loaded 48 test samples
Class distribution in training: [16 16 16 16]
Class distribution in test: [12 12 12 12]
Train batches: 8
Test batches: 6


In [ ]:
cross_dataset = cross_train_loader.dataset
intra_dataset = intra_train_loader.dataset

def build_person_subsets(dataset, source_name):
    """Build subsets of the dataset grouped by person_id."""
    if dataset.person_ids is None:
        raise ValueError(f"{source_name} dataset must be created with add_person_id=True")

    grouped_indices = {}
    for index, person_id in enumerate(dataset.person_ids):
        grouped_indices.setdefault(int(person_id), []).append(index)

    return [
        (f"{source_name}_{person_id}", Subset(dataset, indices), person_id)
        for person_id, indices in sorted(grouped_indices.items())
    ]


cross_person_splits = build_person_subsets(cross_dataset, "cross")
intra_person_splits = build_person_subsets(intra_dataset, "intra")
person_splits = cross_person_splits + intra_person_splits

print("Person-level folds:")
for split_name, subset, person_id in person_splits:
    print(f"  {split_name}: person_id={person_id}, samples={len(subset)}")

Person-level folds:
  cross_113922: person_id=113922, samples=32
  cross_164636: person_id=164636, samples=32
  intra_105923: person_id=105923, samples=32


In [6]:
def cross_validation(model_class, person_splits, batch_size=BATCH_SIZE, epochs=EPOCHS, model_kwargs=None):
    fold_accuracies = []
    model_kwargs = model_kwargs or {}
    use_pin_memory = DEVICE == "cuda"

    for fold, (test_name, test_subset, test_person_id) in enumerate(person_splits):
        train_subsets = [
            subset
            for split_name, subset, _ in person_splits
            if split_name != test_name
        ]

        fold_train_dataset = ConcatDataset(train_subsets)
        fold_train_loader = DataLoader(
            fold_train_dataset,
            batch_size=batch_size,
            shuffle=True,
            pin_memory=use_pin_memory,
        )
        val_loader = DataLoader(
            test_subset,
            batch_size=batch_size,
            shuffle=False,
            pin_memory=use_pin_memory,
        )

        model = model_class(num_classes=NUM_CLASSES, **model_kwargs).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

        print(
            f"Starting fold {fold + 1}/{len(person_splits)} | "
            f"Test person: {test_name} (person_id={test_person_id})"
        )

        for epoch in range(epochs):
            train_loss, train_acc = train_one_epoch(
                model,
                fold_train_loader,
                criterion,
                optimizer,
                DEVICE,
            )
            val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

            print(
                f"Fold {fold + 1} | Epoch {epoch + 1}/{epochs} | "
                f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%"
            )

        fold_accuracies.append(val_acc)

    print(
        f"Mean person-level validation accuracy: {np.mean(fold_accuracies):.2f}%"
    )

In [8]:
MEGGAN_FAST_CONFIG = {
    "temporal_hidden": 32,
    "graph_hidden": 64,
    "num_heads": 2,
    "dropout": 0.15,
}

cross_validation(
    CNNBaseline1D,
    person_splits,
    batch_size=16,
    epochs=20,
)

Starting fold 1/3 | Test person: cross_113922 (person_id=113922)
Fold 1 | Epoch 1/20 | Train Acc: 53.12% | Val Acc: 53.12%
Fold 1 | Epoch 2/20 | Train Acc: 79.69% | Val Acc: 65.62%
Fold 1 | Epoch 3/20 | Train Acc: 84.38% | Val Acc: 62.50%
Fold 1 | Epoch 4/20 | Train Acc: 93.75% | Val Acc: 56.25%
Fold 1 | Epoch 5/20 | Train Acc: 96.88% | Val Acc: 65.62%
Fold 1 | Epoch 6/20 | Train Acc: 96.88% | Val Acc: 75.00%
Fold 1 | Epoch 7/20 | Train Acc: 100.00% | Val Acc: 62.50%
Fold 1 | Epoch 8/20 | Train Acc: 100.00% | Val Acc: 59.38%
Fold 1 | Epoch 9/20 | Train Acc: 100.00% | Val Acc: 75.00%
Fold 1 | Epoch 10/20 | Train Acc: 100.00% | Val Acc: 75.00%
Fold 1 | Epoch 11/20 | Train Acc: 100.00% | Val Acc: 75.00%
Fold 1 | Epoch 12/20 | Train Acc: 100.00% | Val Acc: 75.00%
Fold 1 | Epoch 13/20 | Train Acc: 100.00% | Val Acc: 75.00%
Fold 1 | Epoch 14/20 | Train Acc: 100.00% | Val Acc: 75.00%
Fold 1 | Epoch 15/20 | Train Acc: 96.88% | Val Acc: 78.12%
Fold 1 | Epoch 16/20 | Train Acc: 100.00% | Val Acc

In [ ]:
run_grid_search(cross_train_loader, cross_test_loader, NUM_CLASSES, epochs=15)


Testing parameters:
{'learning_rate': 0.001, 'kernel_size': 3, 'dropout': 0.2, 'hidden_channels': 32}
Epoch 1/15 | Train Acc: 57.81% | Val Acc: 50.00%
Epoch 2/15 | Train Acc: 84.38% | Val Acc: 52.08%
Epoch 3/15 | Train Acc: 87.50% | Val Acc: 70.83%
Epoch 4/15 | Train Acc: 93.75% | Val Acc: 64.58%
Epoch 5/15 | Train Acc: 100.00% | Val Acc: 64.58%
Epoch 6/15 | Train Acc: 100.00% | Val Acc: 58.33%
Epoch 7/15 | Train Acc: 98.44% | Val Acc: 50.00%
Epoch 8/15 | Train Acc: 96.88% | Val Acc: 60.42%
Epoch 9/15 | Train Acc: 100.00% | Val Acc: 64.58%
Epoch 10/15 | Train Acc: 96.88% | Val Acc: 66.67%
Epoch 11/15 | Train Acc: 96.88% | Val Acc: 72.92%
Epoch 12/15 | Train Acc: 95.31% | Val Acc: 66.67%
Epoch 13/15 | Train Acc: 90.62% | Val Acc: 62.50%
Epoch 14/15 | Train Acc: 93.75% | Val Acc: 66.67%
Epoch 15/15 | Train Acc: 95.31% | Val Acc: 62.50%

Testing parameters:
{'learning_rate': 0.001, 'kernel_size': 3, 'dropout': 0.2, 'hidden_channels': 64}
Epoch 1/15 | Train Acc: 43.75% | Val Acc: 45.83%
E

KeyboardInterrupt: 